In [23]:
from datasets import load_dataset
from datasets import Dataset, Features, Value, ClassLabel
from datasets import DatasetDict
import pandas as pd
import evaluate
import numpy as np

import torch
from transformers import AdamW, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding


dataset_train = "/home/jcorvi/git/bsc/etransafe/preclinical-toxicological-corpus/annotated-corpus/2021-05-31/training/final/CORPUS_SENTENCES.txt_dev"
dataset_test =  "/home/jcorvi/git/bsc/etransafe/preclinical-toxicological-corpus/annotated-corpus/2021-05-31/training/final/CORPUS_SENTENCES.txt_test"
colnames=['label', 'text', 'size', 'mask'] 

#dataset = load_dataset("csv", data_files=["my_file_1.csv", "my_file_2.csv", "my_file_3.csv"])

df = pd.read_csv(dataset_train,sep='\t', names=colnames, header=None)
df_test = pd.read_csv(dataset_test,sep='\t', names=colnames, header=None)
df=df[["label", "text"]]
df_test=df_test[["label", "text"]]
#df = pd.DataFrame(df)
#load Dataset from Pandas DataFrame
dataset_train = Dataset.from_pandas(df)
dataset_test = Dataset.from_pandas(df_test)

#dataset = Dataset.from_pandas({"train": df, "test": df_test})

#dataset = load_dataset("csv", sep='\t',data_files={"train": dataset_train, "test": dataset_test})

#dataset = load_dataset("text", data_files={"train": "my_train_file.txt", "test": "my_test_file.txt"}, sample_by="document")

#dataset = dict{"train":dataset,"test":dataset_test}

#raw_datasets = load_dataset("csv", data_files=dataset_dir, delimiter="\t", column_names=["title", "text", "mask","size"])


pretox_class_names = ["NO_PRETOX_REL", "PRETOX_REL"]
pretox_features = Features({'text': Value('string'), 'label': ClassLabel(names=pretox_class_names)})

#load Dataset from Pandas DataFrame
dataset_train = Dataset.from_pandas(df, features=pretox_features)
dataset_test = Dataset.from_pandas(df_test, features=pretox_features)

#dataset = load_dataset('csv', data_files=file_dict, delimiter='\t', column_names=['text', 'label'], features=pretox_features)

raw_dataset = DatasetDict({"train":dataset_train,"validation":dataset_test})
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2049
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 883
    })
})

In [24]:
raw_train_dataset = raw_dataset["train"]
raw_train_dataset[0]

{'text': 'The mean plasma levels of the compound in males at study termination were 1.3, 9.1 and 104.8 ng/ml whilst those in females were 1.4, 5.9 and 85.7 ng/ml at the low, mid and high dose levels, respectively.',
 'label': 0}

In [25]:
raw_train_dataset.features

{'text': Value(dtype='string', id=None),
 'label': ClassLabel(names=['NO_PRETOX_REL', 'PRETOX_REL'], id=None)}

In [26]:

# Same as before
#checkpoint = "bert-base-uncased"
checkpoint = "dmis-lab/biobert-v1.1"


model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)




loading configuration file config.json from cache at /home/jcorvi/.cache/huggingface/hub/models--dmis-lab--biobert-v1.1/snapshots/551ca18efd7f052c8dfa0b01c94c2a8e68bc5488/config.json
Model config BertConfig {
  "_name_or_path": "dmis-lab/biobert-v1.1",
  "architectures": [
    "BertModel"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.25.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}

loading weights file pytorch_model.bin from cache at /home/jcorvi/.cache/huggingface/hub/models--dmis-lab--biobert-v1.1/snapshots/551ca18efd7f052c8dfa0b01c94c

In [ ]:
#revisar esta parte
from transformers import AutoConfig, AutoModelForSequenceClassification
# define the mappings as dictionaries
label2id = {'NO_PRETOX_REL':0, 'PRETOX_REL':1}
id2label = {0:'NO_PRETOX_REL', 1:'PRETOX_REL'}
# define model checkpoint - can be the same model that you already have on the hub
model_ckpt = ...
# define config
config = AutoConfig.from_pretrained(model_ckpt, label2id=label2id, id2label=id2label)
# load model with config
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, config=config)
# export model
model.save_pretrained(target_name_or_path)

In [27]:
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True) #cuidado agregue padding para usar la segunda opcion

In [28]:
tokenized_datasets = raw_dataset.map(tokenize_function, batched=True)
tokenized_datasets

  0%|          | 0/3 [00:00<?, ?ba/s]

  0%|          | 0/1 [00:00<?, ?ba/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2049
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 883
    })
})

In [29]:
tokenized_datasets['train'][0]

{'text': 'The mean plasma levels of the compound in males at study termination were 1.3, 9.1 and 104.8 ng/ml whilst those in females were 1.4, 5.9 and 85.7 ng/ml at the low, mid and high dose levels, respectively.',
 'label': 0,
 'input_ids': [101,
  1109,
  1928,
  13441,
  3001,
  1104,
  1103,
  7090,
  1107,
  3508,
  1120,
  2025,
  20776,
  1127,
  122,
  119,
  124,
  117,
  130,
  119,
  122,
  1105,
  9377,
  119,
  129,
  21174,
  120,
  182,
  1233,
  5994,
  1343,
  1107,
  3032,
  1127,
  122,
  119,
  125,
  117,
  126,
  119,
  130,
  1105,
  4859,
  119,
  128,
  21174,
  120,
  182,
  1233,
  1120,
  1103,
  1822,
  117,
  2286,
  1105,
  1344,
  13753,
  3001,
  117,
  3569,
  119,
  102],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,


In [30]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [31]:
def compute_metrics(eval_pred):
    metric1 = evaluate.load("precision")
    metric2 = evaluate.load("recall")
    metric3 = evaluate.load("accuracy")
    metric4 = evaluate.load("f1")
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision = metric1.compute(predictions=predictions, references=labels)["precision"]
    recall = metric2.compute(predictions=predictions, references=labels)["recall"]
    accuracy = metric3.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric4.compute(predictions=predictions, references=labels)["f1"]
    return {"precision": precision, "recall": recall, "accuracy": accuracy, "f1": f1 }

In [32]:
#using the Trainer Api 
training_args = TrainingArguments("pretox-trainer", evaluation_strategy="epoch")
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],#validation goes here 
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).


In [33]:
trainer.train()

The following columns in the training set don't have a corresponding argument in `BertForSequenceClassification.forward` and have been ignored: text. If text are not expected by `BertForSequenceClassification.forward`,  you can safely ignore this message.
/home/jcorvi/.local/lib/python3.7/site-packages/transformers/optimization.py:310: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  FutureWarning,
***** Running training *****
  Num examples = 2049
  Num Epochs = 3
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 1
  Total optimization steps = 771
  Number of trainable parameters = 108311810
You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a

Epoch,Training Loss,Validation Loss,Precision,Recall,Accuracy,F1
1,No log,0.162417,0.924528,0.975124,0.952435,0.949153
2,0.183700,0.213000,0.939904,0.972637,0.959230,0.955990
3,0.183700,0.232241,0.946602,0.970149,0.961495,0.958231


The following columns in the evaluation set don't have a corresponding argument in `BertForSequenceClassification.forward` and have been ignored: text. If text are not expected by `BertForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 883
  Batch size = 8
Saving model checkpoint to pretox-trainer/checkpoint-500
Configuration saved in pretox-trainer/checkpoint-500/config.json
Model weights saved in pretox-trainer/checkpoint-500/pytorch_model.bin
tokenizer config file saved in pretox-trainer/checkpoint-500/tokenizer_config.json
Special tokens file saved in pretox-trainer/checkpoint-500/special_tokens_map.json
The following columns in the evaluation set don't have a corresponding argument in `BertForSequenceClassification.forward` and have been ignored: text. If text are not expected by `BertForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 883
  Ba

TrainOutput(global_step=771, training_loss=0.13029115333012878, metrics={'train_runtime': 1296.3737, 'train_samples_per_second': 4.742, 'train_steps_per_second': 0.595, 'total_flos': 279513871328820.0, 'train_loss': 0.13029115333012878, 'epoch': 3.0})

In [34]:
#Save the model to path
trainer.save_model("pretox-sentence-classifier-model-transformers-1.0.0")

Saving model checkpoint to pretox-sentence-classifier-model-transformers-1.0.0
Configuration saved in pretox-sentence-classifier-model-transformers-1.0.0/config.json
Model weights saved in pretox-sentence-classifier-model-transformers-1.0.0/pytorch_model.bin
tokenizer config file saved in pretox-sentence-classifier-model-transformers-1.0.0/tokenizer_config.json
Special tokens file saved in pretox-sentence-classifier-model-transformers-1.0.0/special_tokens_map.json
